# Chapter 4: Perception --- Three-Tier Knowledge Extraction

**Learning objectives:**
1. Understand why LLMs fail on structured extraction (FinStructBench motivation)
2. Implement Tier 1 deterministic extraction from tool raw dicts (100% accuracy)
3. Contrast Tier 1 with Tier 2 regex extraction and Tier 3 LLM/text extraction
4. Build a unified `extract_from_tool()` dispatcher with automatic tier fallback
5. Measure extraction accuracy across tiers on identical data

In [ ]:
# === Preamble (all notebooks) ===

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}, CUDA: {torch.cuda.is_available()}")

import pandas as pd

## Motivation: Why LLMs Fail on Structured Extraction

FinStructBench shows that LLMs achieve only **45%** on structured extraction from tabular/financial documents, compared to **100%** for graph-based retrieval. The accuracy degrades along a complexity gradient:

| Task Type | LLM Accuracy | Graph-Based |
|-----------|-------------|-------------|
| Threshold comparison | 86% | 100% |
| Exact recall | 44% | 100% |
| Counting | 34% | 100% |
| Multi-hop reasoning | 0% | 100% |

**Key error modes:**
- Adjacent cell confusion (37%): LLMs read the wrong row/column from a table
- Partial enumeration (68%): LLMs miss items when counting
- Multi-hop decomposition (100%): LLMs fail completely on chains like "find the model with the highest AUC, then look up its robustness score"

This motivates a tiered approach: use **deterministic parsing** (Tier 1) whenever structured data is available, fall back to **regex** (Tier 2) or **LLM** (Tier 3) only when necessary.

In [ ]:
# --- Inspect the three-tier extraction module ---
from knowlytix.harness.governance.extractors import (
    extract_from_tool,
    tier1_extract, tier2_extract, tier3_extract,
    compare_extraction_accuracy,
    _TIER1_EXTRACTORS,
)

print(f"Tier 1 extractors available for {len(_TIER1_EXTRACTORS)} tools:")
for tool_name in _TIER1_EXTRACTORS:
    print(f"  - {tool_name}")

print("\nExtractionResult fields: triples, enm_entries, tier, tool_name, n_facts")
print("ENMEntry fields:        category, entity_id, value")

## Lab 4.1 --- Deterministic Extraction from sklearn Tool Outputs

We train a classifier on Iris using the governed tool set, then extract knowledge via Tier 1.  Because the raw dict contains every metric as a pure Python float, Tier 1 extraction achieves **100% accuracy** with zero ambiguity.

### Step 1: Execute tools and collect raw outputs

In [ ]:
# --- Lab 4.1a: Execute tools, collect both display strings and raw dicts ---
from knowlytix.harness.governance.session import SessionState
from knowlytix.harness.governance.tools import execute_tool

state = SessionState()

# 1. Load dataset
display_load, raw_load = execute_tool(
    "load_dataset", {"name": "iris", "builtin": "iris"}, state
)
print("=== load_dataset ===")
print(f"Display: {display_load}")
print(f"Raw keys: {list(raw_load.keys())}")

# 2. Train a model
display_train, raw_train = execute_tool(
    "train_model",
    {"dataset": "iris", "model_type": "RandomForestClassifier",
     "model_name": "rf_100", "params": {"n_estimators": 100, "random_state": 42}},
    state,
)
print("\n=== train_model ===")
print(f"Display: {display_train}")
print(f"Raw: {raw_train}")

In [4]:
# --- Lab 4.1a (continued): Run diagnostic tools ---

# 3. Diagnose robustness
display_rob, raw_rob = execute_tool(
    "diagnose_robustness",
    {"dataset": "iris", "model_name": "rf_100",
     "noise_levels": [0.01, 0.05, 0.1, 0.2]},
    state,
)
print("=== diagnose_robustness ===")
print(f"Display:\n{display_rob}")
print(f"Raw: {raw_rob}")

# 4. Explain feature importance
display_imp, raw_imp = execute_tool(
    "explain_importance",
    {"dataset": "iris", "model_name": "rf_100", "n_repeats": 10},
    state,
)
print("\n=== explain_importance ===")
print(f"Display:\n{display_imp}")
print(f"Raw importances: {raw_imp.get('importances', {})}")

=== diagnose_robustness ===
Display:
Robustness diagnosis for 'rf_100':
  noise_0.01: {'accuracy': 1.0}
  noise_0.05: {'accuracy': 1.0}
  noise_0.1: {'accuracy': 1.0}
  noise_0.2: {'accuracy': 0.9333}
Raw: {'model_name': 'rf_100', 'noise_results': {'noise_0.01': {'accuracy': 1.0}, 'noise_0.05': {'accuracy': 1.0}, 'noise_0.1': {'accuracy': 1.0}, 'noise_0.2': {'accuracy': 0.9333}}}



=== explain_importance ===
Display:
Permutation importance for 'rf_100':
  petal length (cm): 0.2500
  petal width (cm): 0.2200
  sepal length (cm): 0.0000
  sepal width (cm): 0.0000
Raw importances: {'petal length (cm)': 0.25, 'petal width (cm)': 0.22, 'sepal length (cm)': 0.0, 'sepal width (cm)': 0.0}


### Step 2: Tier 1 extraction --- deterministic parsing of raw dicts

Each tool returns a `(display_string, raw_dict)` pair. Tier 1 parses the raw dict directly into triples and ENM entries. No regex, no LLM, no ambiguity.

In [5]:
# --- Lab 4.1b: Tier 1 extraction on all four tool outputs ---

tool_outputs = [
    ("load_dataset",        {"name": "iris", "builtin": "iris"},
     raw_load,  display_load),
    ("train_model",         {"dataset": "iris", "model_type": "RandomForestClassifier",
                             "model_name": "rf_100"},
     raw_train, display_train),
    ("diagnose_robustness", {"dataset": "iris", "model_name": "rf_100"},
     raw_rob,   display_rob),
    ("explain_importance",  {"dataset": "iris", "model_name": "rf_100"},
     raw_imp,   display_imp),
]

tier1_results = {}
for tool_name, args, raw, display in tool_outputs:
    result = tier1_extract(tool_name, args, raw)
    tier1_results[tool_name] = result
    print(f"=== {tool_name} (Tier 1) ===")
    print(f"  Triples:     {len(result.triples)}")
    print(f"  ENM entries: {len(result.enm_entries)}")
    print(f"  Total facts: {result.n_facts}")
    # Show first 3 triples
    for t in result.triples[:3]:
        val = f" [{t.numeric_value}]" if t.numeric_value is not None else ""
        print(f"    ({t.head}, {t.relation}, {t.tail}){val}")
    if len(result.triples) > 3:
        print(f"    ... and {len(result.triples) - 3} more triples")
    print()

=== load_dataset (Tier 1) ===
  Triples:     9
  ENM entries: 4
  Total facts: 13
    (iris, is_type, dataset)
    (iris, has_task_type, classification)
    (iris, has_target, target)
    ... and 6 more triples

=== train_model (Tier 1) ===
  Triples:     9
  ENM entries: 7
  Total facts: 16
    (rf_100, is_type, RandomForestClassifier)
    (rf_100, trained_on, iris)
    (rf_100, has_train_accuracy, 1.0) [1.0]
    ... and 6 more triples

=== diagnose_robustness (Tier 1) ===
  Triples:     7
  ENM entries: 4
  Total facts: 11
    (rf_100, diagnosed_for, robustness)
    (rf_100, robustness_noise_0.01_accuracy, 1.0) [1.0]
    (rf_100, robustness_noise_0.05_accuracy, 1.0) [1.0]
    ... and 4 more triples

=== explain_importance (Tier 1) ===
  Triples:     6
  ENM entries: 4
  Total facts: 10
    (rf_100, diagnosed_for, feature_importance)
    (rf_100, importance_of_petal length (cm), 0.25) [0.25]
    (rf_100, importance_of_petal width (cm), 0.22) [0.22]
    ... and 3 more triples



In [6]:
# --- Lab 4.1c: Verify Tier 1 accuracy against ground truth ---

# Build ground truth from the raw dicts (which we KNOW are correct
# because they come from sklearn computations, not LLM parsing).

# Ground truth for train_model
gt_train = {}
for k, v in raw_train["metrics"].items():
    gt_train[k] = float(v)

result_t1 = tier1_results["train_model"]
acc_report = compare_extraction_accuracy(gt_train, result_t1)

print("=== Tier 1 Accuracy: train_model ===")
print(f"  Ground truth metrics: {len(gt_train)}")
print(f"  Matched:    {acc_report['matched']}")
print(f"  Mismatched: {acc_report['mismatched']}")
print(f"  Missing:    {acc_report['missing']}")
print(f"  Accuracy:   {acc_report['accuracy']:.1%}")
print()
for key, gt_val, ex_val, status in acc_report["details"]:
    print(f"  {key:<25s}  GT={gt_val}  Extracted={ex_val}  [{status}]")

=== Tier 1 Accuracy: train_model ===
  Ground truth metrics: 7
  Matched:    7
  Mismatched: 0
  Missing:    0
  Accuracy:   100.0%

  train_accuracy             GT=1.0  Extracted=1.0  [MATCH]
  train_f1_macro             GT=1.0  Extracted=1.0  [MATCH]
  train_auc_ovr              GT=1.0  Extracted=1.0  [MATCH]
  test_accuracy              GT=1.0  Extracted=1.0  [MATCH]
  test_f1_macro              GT=1.0  Extracted=1.0  [MATCH]
  test_auc_ovr               GT=1.0  Extracted=1.0  [MATCH]
  train_test_gap             GT=0.0  Extracted=0.0  [MATCH]


In [7]:
# --- Lab 4.1d: Verify Tier 1 accuracy for robustness extraction ---
gt_rob = {}
for noise_key, noise_vals in raw_rob["noise_results"].items():
    for metric_name, metric_val in noise_vals.items():
        gt_rob[f"{noise_key}_{metric_name}"] = float(metric_val)

result_rob = tier1_results["diagnose_robustness"]
acc_rob = compare_extraction_accuracy(gt_rob, result_rob)

print("=== Tier 1 Accuracy: diagnose_robustness ===")
print(f"  Ground truth metrics: {len(gt_rob)}")
print(f"  Matched:    {acc_rob['matched']}")
print(f"  Mismatched: {acc_rob['mismatched']}")
print(f"  Missing:    {acc_rob['missing']}")
print(f"  Accuracy:   {acc_rob['accuracy']:.1%}")
print()

# Verify importance extraction
gt_imp = {f"pfi_{k}": v for k, v in raw_imp["importances"].items()}
result_imp = tier1_results["explain_importance"]
acc_imp = compare_extraction_accuracy(gt_imp, result_imp)

print("=== Tier 1 Accuracy: explain_importance ===")
print(f"  Ground truth metrics: {len(gt_imp)}")
print(f"  Matched:    {acc_imp['matched']}")
print(f"  Accuracy:   {acc_imp['accuracy']:.1%}")

# Overall Tier 1 summary
total_gt = len(gt_train) + len(gt_rob) + len(gt_imp)
total_match = acc_report['matched'] + acc_rob['matched'] + acc_imp['matched']
print(f"\n{'='*50}")
print(f"OVERALL TIER 1 ACCURACY: {total_match}/{total_gt} = {total_match/total_gt:.1%}")
print(f"{'='*50}")

=== Tier 1 Accuracy: diagnose_robustness ===
  Ground truth metrics: 4
  Matched:    4
  Mismatched: 0
  Missing:    0
  Accuracy:   100.0%

=== Tier 1 Accuracy: explain_importance ===
  Ground truth metrics: 4
  Matched:    4
  Accuracy:   100.0%

OVERALL TIER 1 ACCURACY: 15/15 = 100.0%


## Lab 4.2 --- Tier Comparison: Deterministic vs Regex vs Text Parsing

Now we run the **same data** through all three tiers and compare extraction accuracy. This demonstrates the reliability gradient: Tier 1 is perfect, Tier 2 loses information, and Tier 3 is the noisiest.

### Tier 2: Regex extraction from display strings

Tier 2 uses regular expressions to parse key-value pairs from the human-readable display string. This is fragile: formatting changes break extraction and multi-line structures are hard to parse.

In [8]:
# --- Lab 4.2a: Tier 2 regex extraction on train_model display string ---

print("Display string passed to Tier 2:")
print("-" * 50)
print(display_train)
print("-" * 50)

result_t2 = tier2_extract(
    "train_model",
    {"model_name": "rf_100", "dataset": "iris"},
    display_train,
)

print(f"\nTier 2 extracted {result_t2.n_facts} facts:")
print(f"  Triples:     {len(result_t2.triples)}")
print(f"  ENM entries: {len(result_t2.enm_entries)}")
for t in result_t2.triples:
    val = f" [{t.numeric_value}]" if t.numeric_value is not None else ""
    print(f"    ({t.head}, {t.relation}, {t.tail}){val}")

# Compare Tier 2 accuracy against ground truth
acc_t2 = compare_extraction_accuracy(gt_train, result_t2)
print(f"\nTier 2 accuracy on train_model: {acc_t2['accuracy']:.1%}")
print(f"  Matched: {acc_t2['matched']}, Missing: {acc_t2['missing']}")

Display string passed to Tier 2:
--------------------------------------------------
Trained 'rf_100' (RandomForestClassifier):
  train_accuracy: 1.0
  train_f1_macro: 1.0
  train_auc_ovr: 1.0
  test_accuracy: 1.0
  test_f1_macro: 1.0
  test_auc_ovr: 1.0
  train_test_gap: 0.0
--------------------------------------------------

Tier 2 extracted 14 facts:
  Triples:     7
  ENM entries: 7
    (rf_100, has_train_accuracy, 1.0) [1.0]
    (rf_100, has_train_f1_macro, 1.0) [1.0]
    (rf_100, has_train_auc_ovr, 1.0) [1.0]
    (rf_100, has_test_accuracy, 1.0) [1.0]
    (rf_100, has_test_f1_macro, 1.0) [1.0]
    (rf_100, has_test_auc_ovr, 1.0) [1.0]
    (rf_100, has_train_test_gap, 0.0) [0.0]

Tier 2 accuracy on train_model: 100.0%
  Matched: 7, Missing: 0


In [9]:
# --- Lab 4.2b: Tier 3 text extraction on the same display string ---
# Tier 3 in production uses an LLM. Here we use extract_triples_simple()
# which looks for "TRIPLE:" formatted lines. Since the display string
# is NOT in TRIPLE: format, Tier 3 extracts nothing --- demonstrating
# that unstructured display strings defeat text parsers.

result_t3 = tier3_extract(
    "train_model",
    {"model_name": "rf_100", "dataset": "iris"},
    display_train,
)

print(f"Tier 3 extracted {result_t3.n_facts} facts from display string")
print("  (Display string has no TRIPLE: lines, so simple parser finds nothing)")

# Now try Tier 3 on properly formatted text (simulating LLM output)
formatted_text = ""
for k, v in raw_train["metrics"].items():
    formatted_text += f"TRIPLE: rf_100 | has_{k} | {v} | {v}\n"

print("\nFormatted text for Tier 3:")
print(formatted_text)

result_t3_fmt = tier3_extract(
    "train_model",
    {"model_name": "rf_100", "dataset": "iris"},
    formatted_text,
)

print(f"Tier 3 on formatted text: {result_t3_fmt.n_facts} facts")
acc_t3 = compare_extraction_accuracy(gt_train, result_t3_fmt)
print(f"Tier 3 accuracy (formatted): {acc_t3['accuracy']:.1%}")

Tier 3 extracted 0 facts from display string
  (Display string has no TRIPLE: lines, so simple parser finds nothing)

Formatted text for Tier 3:
TRIPLE: rf_100 | has_train_accuracy | 1.0 | 1.0
TRIPLE: rf_100 | has_train_f1_macro | 1.0 | 1.0
TRIPLE: rf_100 | has_train_auc_ovr | 1.0 | 1.0
TRIPLE: rf_100 | has_test_accuracy | 1.0 | 1.0
TRIPLE: rf_100 | has_test_f1_macro | 1.0 | 1.0
TRIPLE: rf_100 | has_test_auc_ovr | 1.0 | 1.0
TRIPLE: rf_100 | has_train_test_gap | 0.0 | 0.0

Tier 3 on formatted text: 14 facts
Tier 3 accuracy (formatted): 100.0%


In [10]:
# --- Lab 4.2c: Head-to-head comparison table ---

# Run all three tiers across all tool outputs and tally accuracy
comparison_rows = []

for tool_name, args, raw, display in tool_outputs:
    # Build ground truth based on tool type
    if tool_name == "load_dataset":
        gt = {"n_samples": float(raw["n_samples"]),
              "n_features": float(raw["n_features"]),
              "train_size": float(raw["train_size"]),
              "test_size": float(raw["test_size"])}
    elif tool_name == "train_model":
        gt = {k: float(v) for k, v in raw["metrics"].items()}
    elif tool_name == "diagnose_robustness":
        gt = {}
        for nk, nv in raw["noise_results"].items():
            for mk, mv in nv.items():
                gt[f"{nk}_{mk}"] = float(mv)
    elif tool_name == "explain_importance":
        gt = {f"pfi_{k}": v for k, v in raw["importances"].items()}
    else:
        gt = {}

    # Tier 1
    r1 = tier1_extract(tool_name, args, raw)
    a1 = compare_extraction_accuracy(gt, r1) if gt else {"accuracy": 1.0, "total": 0}

    # Tier 2
    r2 = tier2_extract(tool_name, args, display)
    a2 = compare_extraction_accuracy(gt, r2) if gt else {"accuracy": 0.0, "total": 0}

    # Tier 3 (on raw display, not formatted)
    r3 = tier3_extract(tool_name, args, display)
    a3 = compare_extraction_accuracy(gt, r3) if gt else {"accuracy": 0.0, "total": 0}

    comparison_rows.append({
        "Tool": tool_name,
        "GT Facts": a1.get("total", 0),
        "Tier 1 Acc": f"{a1['accuracy']:.0%}",
        "Tier 1 Facts": r1.n_facts if r1 else 0,
        "Tier 2 Acc": f"{a2['accuracy']:.0%}",
        "Tier 2 Facts": r2.n_facts,
        "Tier 3 Acc": f"{a3['accuracy']:.0%}",
        "Tier 3 Facts": r3.n_facts,
    })

comp_df = pd.DataFrame(comparison_rows)
print("Three-Tier Extraction Accuracy Comparison")
print("=" * 75)
print(comp_df.to_string(index=False))

Three-Tier Extraction Accuracy Comparison
               Tool  GT Facts Tier 1 Acc  Tier 1 Facts Tier 2 Acc  Tier 2 Facts Tier 3 Acc  Tier 3 Facts
       load_dataset         4       100%            13         0%             4         0%             0
        train_model         7       100%            16       100%            14         0%             0
diagnose_robustness         4       100%            11         0%             0         0%             0
 explain_importance         4       100%            10         0%             0         0%             0


### The Unified Dispatcher: `extract_from_tool()`

In production the agent does not choose a tier manually. The `extract_from_tool()` function tries Tier 1 first, falls back to Tier 2 if Tier 1 has no extractor for the tool, then falls back to Tier 3.

In [11]:
# --- Lab 4.2d: Unified dispatcher in action ---

print("Unified extract_from_tool() dispatcher:")
print("=" * 60)

for tool_name, args, raw, display in tool_outputs:
    result = extract_from_tool(tool_name, args, raw, display)
    print(f"  {tool_name:<25s}  -> Tier {result.tier}  "
          f"({result.n_facts} facts, "
          f"{len(result.triples)} triples, "
          f"{len(result.enm_entries)} ENM)")

# Also show what happens when Tier 1 is unavailable
# (tool not in _TIER1_EXTRACTORS)
print("\nFallback behavior for unsupported tools:")
result_stub = extract_from_tool(
    "recall_memory",
    {"entity": "rf_100"},
    None,
    "Memory recall: rf_100 has accuracy 0.9667",
)
print(f"  recall_memory -> Tier {result_stub.tier} "
      f"({result_stub.n_facts} facts)")
for t in result_stub.triples:
    val = f" [{t.numeric_value}]" if t.numeric_value is not None else ""
    print(f"    ({t.head}, {t.relation}, {t.tail}){val}")

Unified extract_from_tool() dispatcher:
  load_dataset               -> Tier 1  (13 facts, 9 triples, 4 ENM)
  train_model                -> Tier 1  (16 facts, 9 triples, 7 ENM)
  diagnose_robustness        -> Tier 1  (11 facts, 7 triples, 4 ENM)
  explain_importance         -> Tier 1  (10 facts, 6 triples, 4 ENM)

Fallback behavior for unsupported tools:
  recall_memory -> Tier 3 (0 facts)


## Lab 4.2e --- Leaderboard and Fairness Extraction

We train a second model, build a leaderboard and run fairness diagnosis to exercise additional Tier 1 extractors.

In [12]:
# --- Lab 4.2e: Train a second model and build leaderboard ---

display_dt, raw_dt = execute_tool(
    "train_model",
    {"dataset": "iris", "model_type": "DecisionTreeClassifier",
     "model_name": "dt_5", "params": {"max_depth": 5, "random_state": 42}},
    state,
)
print("Trained DecisionTreeClassifier:")
print(display_dt)

# Leaderboard
display_lb, raw_lb = execute_tool(
    "leaderboard", {"dataset": "iris"}, state,
)
print(f"\n{display_lb}")

# Tier 1 extraction on leaderboard
result_lb = extract_from_tool("leaderboard", {"dataset": "iris"},
                               raw_lb, display_lb)
print(f"\nLeaderboard extraction: Tier {result_lb.tier}, "
      f"{result_lb.n_facts} facts")
for t in result_lb.triples[:6]:
    val = f" [{t.numeric_value}]" if t.numeric_value is not None else ""
    print(f"  ({t.head}, {t.relation}, {t.tail}){val}")
if len(result_lb.triples) > 6:
    print(f"  ... and {len(result_lb.triples) - 6} more")

Trained DecisionTreeClassifier:
Trained 'dt_5' (DecisionTreeClassifier):
  train_accuracy: 0.9917
  train_f1_macro: 0.9917
  train_auc_ovr: 0.9998
  test_accuracy: 1.0
  test_f1_macro: 1.0
  test_auc_ovr: 1.0
  train_test_gap: -0.0083

Leaderboard:
Rank  Model                     Type                           test_accuracy  
------------------------------------------------------------------------------
1     rf_100                    RandomForestClassifier         1.0
2     dt_5                      DecisionTreeClassifier         1.0

Leaderboard extraction: Tier 1, 35 facts
  (rf_100, is_type, RandomForestClassifier)
  (rf_100, has_leaderboard_rank, 1) [1.0]
  (rf_100, has_train_accuracy, 1.0) [1.0]
  (rf_100, has_train_f1_macro, 1.0) [1.0]
  (rf_100, has_train_auc_ovr, 1.0) [1.0]
  (rf_100, has_test_accuracy, 1.0) [1.0]
  ... and 13 more


In [13]:
# --- Lab 4.2f: Fairness extraction on breast_cancer dataset ---
# Load a binary classification dataset with a natural split attribute

state2 = SessionState()
execute_tool("load_dataset",
             {"name": "bc", "builtin": "breast_cancer"}, state2)

# Train a model
execute_tool("train_model",
             {"dataset": "bc", "model_type": "RandomForestClassifier",
              "model_name": "rf_bc",
              "params": {"n_estimators": 100, "random_state": 42}},
             state2)

# Use "mean radius" > median as a synthetic protected attribute for demo
# (breast_cancer has no demographic attributes, so we split on a feature)
ds = state2.datasets["bc"]
median_radius = ds["X_test"]["mean radius"].median()
ds["df"]["radius_group"] = (ds["df"]["mean radius"] > median_radius).astype(int)
ds["X_test"]["radius_group"] = (ds["X_test"]["mean radius"] > median_radius).astype(int)

display_fair, raw_fair = execute_tool(
    "diagnose_fairness",
    {"dataset": "bc", "model_name": "rf_bc",
     "protected_attribute": "radius_group"},
    state2,
)
print(display_fair)

# Extract
result_fair = extract_from_tool(
    "diagnose_fairness",
    {"dataset": "bc", "model_name": "rf_bc",
     "protected_attribute": "radius_group"},
    raw_fair, display_fair,
)
print(f"\nFairness extraction: Tier {result_fair.tier}, "
      f"{result_fair.n_facts} facts")
for t in result_fair.triples:
    val = f" [{t.numeric_value}]" if t.numeric_value is not None else ""
    print(f"  ({t.head}, {t.relation}, {t.tail}){val}")

Fairness diagnosis for 'rf_bc' on 'radius_group':
  Group positive rates: {'0': 0.9649, '1': 0.3158}
  Group accuracy:       {'0': 0.9825, '1': 0.9474}
  Adverse Impact Ratio: 0.3273
  Demographic parity diff: 0.6491
  AIR >= 0.8 (4/5 rule): False

Fairness extraction: Tier 1, 11 facts
  (rf_bc, diagnosed_for, fairness)
  (rf_bc, protected_attribute, radius_group)
  (rf_bc, has_air, 0.3273) [0.3273]
  (rf_bc, has_dp_diff, 0.6491) [0.6491]
  (rf_bc, passes_4_5_rule, False)
  (rf_bc, selection_rate_0, 0.9649) [0.9649]
  (rf_bc, selection_rate_1, 0.3158) [0.3158]


## Summary: Extraction Accuracy by Tier

| Tier | Method | Accuracy | Strengths | Weaknesses |
|------|--------|----------|-----------|------------|
| **1** | Deterministic dict parsing | 100% | Zero ambiguity, zero latency, zero cost | Requires per-tool extractor |
| **2** | Regex on display string | Variable (30--80%) | No per-tool code needed | Fragile to format changes |
| **3** | LLM / text parser | Variable (0--100%) | Handles free text | Expensive, non-deterministic, noisy |

**Design principle:** Always prefer the highest-numbered tier that is available. The `extract_from_tool()` dispatcher enforces this automatically.

## Exercises

### Exercise 4.1
Implement a Tier 1 extractor for the `data_summary` tool. Verify 100% accuracy on the Iris dataset summary.

### Exercise 4.2
Build a Tier 2 regex extractor specifically for the leaderboard markdown table format. Test it on a 5-model leaderboard.

### Exercise 4.3
Run the fairness extraction three ways (Tier 1 from raw dict, Tier 2 from display string, Tier 3 from formatted TRIPLE: text). Compare the number of facts extracted and the accuracy of each.